# NicheNet's Ligand Activity Analysis on a Gene Set of Interest

This notebook follows the steps described in the step-by-step analysis with two major differences: a predefined gene set of interest is given, and a different definition of expressed genes.

Here, we explore intercellular communication in the tumor microenvironment of head and neck squamous cell carcinoma (HNSCC) (Puram et al., 2017). We look at which ligands expressed by cancer-associated fibroblasts (CAFs) can induce a specific gene program in neighboring malignant cells -- the partial epithelial-mesenchymal transition (p-EMT) program linked to metastasis.

## Prepare NicheNet Analysis

### Load packages

In [ ]:
import os
os.environ.setdefault("NICHENETR_DATA_DIR", "path/to/nichenetr_data")

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import nichenetr as nn

### Read in NicheNet's networks

In [ ]:
organism = "human"

lr_network = nn.load_lr_network(organism)
ligand_target_matrix = nn.load_ligand_target_matrix(organism)
weighted_networks = nn.load_weighted_networks(organism)

lr_network = lr_network[["from", "to"]].drop_duplicates()
print(lr_network.head())
print(f"\nLigand-target matrix: {ligand_target_matrix.data.shape}")
print(weighted_networks["lr_sig"].head())
print(weighted_networks["gr"].head())

### Read in the expression data of interacting cells

This is publicly available single-cell data from CAF and malignant cells from HNSCC tumors.

In [ ]:
hnscc_data = nn.load_hnscc_expression()
expression = hnscc_data["expression"]
sample_info = hnscc_data["sample_info"]

# Convert expression to a dense DataFrame for convenience
expression_df = pd.DataFrame(
    expression.data.toarray(),
    index=expression.rownames,
    columns=expression.colnames,
)

# Convert aliases to official symbols
expression_df.columns = nn.convert_alias_to_symbols(
    list(expression_df.columns), "human", verbose=False
)

## 1. Define a set of potential ligands

We use the **sender-focused** approach: CAFs as senders and malignant cells as receivers. We define expressed genes using the original authors' definition: aggregate expression $E_a(i) = \log_2(\text{mean}(\text{TPM}(i)_{1..k}) + 1) \geq 4$.

In [ ]:
tumors_remove = ["HN10", "HN", "HN12", "HN13", "HN24", "HN7", "HN8", "HN23"]

CAF_ids = sample_info[
    (sample_info["Lymph node"] == 0)
    & (~sample_info["tumor"].isin(tumors_remove))
    & (sample_info["non-cancer cell type"] == "CAF")
]["cell"].tolist()

malignant_ids = sample_info[
    (sample_info["Lymph node"] == 0)
    & (~sample_info["tumor"].isin(tumors_remove))
    & (sample_info["classified  as cancer cell"] == 1)
]["cell"].tolist()

def get_expressed_custom(expression_df, cell_ids):
    """Compute expressed genes using the Puram et al. definition."""
    sub = expression_df.loc[expression_df.index.isin(cell_ids)]
    # Convert from log2(TPM/10 + 1) back to TPM, then re-aggregate
    tpm = 10 * (2 ** sub - 1)
    agg = np.log2(tpm.mean(axis=0) + 1)
    return agg[agg >= 4].index.tolist()

expressed_genes_sender = get_expressed_custom(expression_df, CAF_ids)
expressed_genes_receiver = get_expressed_custom(expression_df, malignant_ids)

print(f"Expressed genes in CAFs (sender): {len(expressed_genes_sender)}")
print(f"Expressed genes in malignant cells (receiver): {len(expressed_genes_receiver)}")

In [ ]:
# Define potential ligands: expressed in sender, whose cognate receptor is expressed in receiver
ligands = lr_network["from"].unique().tolist()
expressed_ligands = list(set(ligands) & set(expressed_genes_sender))

receptors = lr_network["to"].unique().tolist()
expressed_receptors = list(set(receptors) & set(expressed_genes_receiver))

potential_ligands = (
    lr_network[
        lr_network["from"].isin(expressed_ligands)
        & lr_network["to"].isin(expressed_receptors)
    ]["from"]
    .unique()
    .tolist()
)

print(f"Potential ligands: {len(potential_ligands)}")
print(potential_ligands[:10])

## 2. Define the gene set of interest and background genes

We use the p-EMT gene set defined by Puram et al. as the gene set of interest.

In [ ]:
pemt_signature = nn.load_pemt_signature()
geneset_oi = [g for g in pemt_signature if g in ligand_target_matrix.rownames]
print(f"p-EMT gene set size: {len(geneset_oi)}")

In [ ]:
# Background: all expressed genes in receiver (malignant) cells
background_expressed_genes = [
    g for g in expressed_genes_receiver if g in ligand_target_matrix.rownames
]
print(f"Background expressed genes: {len(background_expressed_genes)}")

## 4. Perform NicheNet ligand activity analysis

Assess how well each CAF-ligand can predict the p-EMT gene set compared to the background.

In [ ]:
ligand_activities = nn.predict_ligand_activities(
    geneset=geneset_oi,
    background_expressed_genes=background_expressed_genes,
    ligand_target_matrix=ligand_target_matrix,
    potential_ligands=potential_ligands,
)

ligand_activities = ligand_activities.sort_values("aupr_corrected", ascending=False)
ligand_activities["rank"] = ligand_activities["aupr_corrected"].rank(ascending=False)
ligand_activities.head(30)

In [ ]:
best_upstream_ligands = (
    ligand_activities.nlargest(30, "aupr_corrected")["test_ligand"].tolist()
)
print(best_upstream_ligands)

## 5. Infer target genes and receptors of top-ranked ligands

In [ ]:
# Active ligand-target links
active_ligand_target_links_df = pd.concat(
    [
        nn.get_weighted_ligand_target_links(
            ligand_oi=lig,
            geneset=geneset_oi,
            ligand_target_matrix=ligand_target_matrix,
            n=200,
        )
        for lig in best_upstream_ligands
    ],
    ignore_index=True,
).dropna()

active_ligand_target_links = nn.prepare_ligand_target_visualization(
    ligand_target_df=active_ligand_target_links_df,
    ligand_target_matrix=ligand_target_matrix,
    cutoff=0.25,
)

order_ligands = [l for l in reversed(best_upstream_ligands) if l in active_ligand_target_links.colnames]
order_targets = [
    t for t in active_ligand_target_links_df["target"].unique()
    if t in active_ligand_target_links.rownames
]

df_full = pd.DataFrame(
    active_ligand_target_links,
    index=active_ligand_target_links.rownames,
    columns=active_ligand_target_links.colnames,
)
vis_ligand_target = df_full.loc[order_targets, order_ligands].T

nn.make_heatmap_ggplot(
    vis_ligand_target,
    y_name="Prioritized CAF-ligands",
    x_name="p-EMT genes in malignant cells",
    color="purple",
    legend_title="Regulatory potential",
    figsize=(10, 8),
    show=True,
)

In [ ]:
# Ligand-receptor heatmap
ligand_receptor_links_df = nn.get_weighted_ligand_receptor_links(
    best_upstream_ligands, expressed_receptors, lr_network, weighted_networks["lr_sig"]
)

vis_ligand_receptor_network = nn.prepare_ligand_receptor_visualization(
    ligand_receptor_links_df, best_upstream_ligands, order_hclust="both"
)

df_lr = pd.DataFrame(
    vis_ligand_receptor_network,
    index=vis_ligand_receptor_network.rownames,
    columns=vis_ligand_receptor_network.colnames,
).T

nn.make_heatmap_ggplot(
    df_lr,
    y_name="Prioritized CAF-ligands",
    x_name="Receptors expressed by malignant cells",
    color="mediumvioletred",
    legend_title="Prior interaction potential",
    figsize=(10, 8),
    show=True,
)

## 6. Summary visualizations

### Ligand activity heatmap

In [ ]:
vis_ligand_aupr = (
    ligand_activities[ligand_activities["test_ligand"].isin(best_upstream_ligands)]
    .set_index("test_ligand")[["aupr_corrected"]]
    .sort_values("aupr_corrected")
)

nn.make_heatmap_ggplot(
    vis_ligand_aupr,
    y_name="Prioritized CAF-ligands",
    x_name="Ligand activity",
    color="darkorange",
    legend_title="AUPR",
    x_axis=False,
    figsize=(3, 8),
    show=True,
)

### Expression of ligands in CAFs per tumor

In [ ]:
# Compute average expression of top ligands per tumor in CAFs
caf_expr = expression_df.loc[expression_df.index.isin(CAF_ids), best_upstream_ligands]
caf_expr = caf_expr.join(sample_info.set_index("cell")["tumor"])

aggregated_caf = caf_expr.groupby("tumor").mean()

# Tumors ordered by p-EMT score (from Puram et al.)
order_tumors = ["HN6", "HN20", "HN26", "HN28", "HN22", "HN25", "HN5", "HN18", "HN17", "HN16"]
order_tumors = [t for t in order_tumors if t in aggregated_caf.index]

vis_ligand_tumor = aggregated_caf.loc[order_tumors, list(reversed(best_upstream_ligands))].T

nn.make_heatmap_ggplot(
    vis_ligand_tumor,
    y_name="Prioritized CAF-ligands",
    x_name="Tumor",
    color="red",
    legend_title="Expression\n(avg over cells)",
    figsize=(8, 8),
    show=True,
)

### Expression of target genes in malignant cells per tumor

In [ ]:
mal_expr = expression_df.loc[expression_df.index.isin(malignant_ids), geneset_oi]
mal_expr = mal_expr.join(sample_info.set_index("cell")["tumor"])

aggregated_mal = mal_expr.groupby("tumor").mean()
vis_target_tumor = aggregated_mal.loc[
    [t for t in order_tumors if t in aggregated_mal.index],
    [t for t in order_targets if t in aggregated_mal.columns],
]

# Quantile scaling
vis_target_scaled = nn.scale_quantile(vis_target_tumor.values)
vis_target_scaled_df = pd.DataFrame(
    vis_target_scaled, index=vis_target_tumor.index, columns=vis_target_tumor.columns
)

nn.make_threecolor_heatmap_ggplot(
    vis_target_scaled_df,
    y_name="Tumor",
    x_name="Target",
    low_color="blue",
    mid_color="whitesmoke",
    mid=0.5,
    high_color="red",
    legend_title="Scaled expression\n(avg over cells)",
    figsize=(10, 5),
    show=True,
)